# Week 4: PLS vs Ridge / LR (LORO)

Supervised compression of AlphaEarth `A00`–`A63` with **Partial Least Squares**, compared to the week-4 balanced pure-cell baselines on the same LORO folds.

AE bands are **not orthogonal** (mean pairwise |corr| ≈ 0.22; effective rank ≈ 10). PLS scores **are** orthogonal by construction — the right place to look for a minimal set of important dimensions.

**Data.** `datasets/week4_pixel_gt_balanced/pixels.parquet` — 1,751,814 pure RES/NRES cells (1:1 per metro).

**Models**
| Family | What |
|---|---|
| `baseline_lr` / `baseline_ridge` | StandardScaler + LR / Ridge on all 64 bands |
| `pls_da_lr` | PLS on binary \(y\) → LR on scores (PLS-DA) |
| `pls_frac` | PLSRegression \(X \to\) `(frac_res, frac_nres, frac_empty)` |
| `pls_frac_lr` | PLS on fractions → LR on scores |
| `pca_lr` | PCA → LR (unsupervised control) |

Coarse sweep `n_components ∈ [2, 4, 8, 12, 16, 24, 32]`; fine PLS-DA sweep `k=1..20` for the minimum orthogonal set. Backend: **gpu** · 8 GPUs / 80 CPUs · 236.6 s.

```bash
.venv/bin/python -u scripts/train_week4_pls_loro.py --backend gpu
.venv/bin/python -u scripts/fine_week4_pls_k_sweep.py
.venv/bin/python -u scripts/build_week4_pls_notebook.py
```


## Headline

| Model | LORO macro-F1 / MAE | OOF |
|---|---:|---:|
| Baseline LR (64-D) | F1 0.779 ± 0.085 | 0.805 |
| PLS-DA → LR (k=32) | F1 0.775 ± 0.088 | 0.804 |
| PLS frac → LR (k=24) | F1 0.775 ± 0.084 | — |
| PCA → LR (k=32) | F1 0.759 ± 0.084 | — |
| Baseline Ridge | MAE 0.263 ± 0.027 | 0.254 |
| PLS frac (k=12) | MAE 0.260 ± 0.028 | 0.253 |

Δ OOF F1 (PLS-DA − baseline) = **-0.0013** ·
Δ OOF MAE (PLS − Ridge) = **-0.0008**


![F1 vs n_components](../../outputs/week4_pls_loro/figures/f1_vs_ncomp.png)

![MAE vs n_components](../../outputs/week4_pls_loro/figures/mae_vs_ncomp.png)

![Best-family F1](../../outputs/week4_pls_loro/figures/best_f1_compare.png)

![Best MAE](../../outputs/week4_pls_loro/figures/best_mae_compare.png)

![Per-metro PLS vs baseline](../../outputs/week4_pls_loro/figures/loro_f1_pls_vs_baseline.png)

![OOF confusion](../../outputs/week4_pls_loro/figures/oof_confusion_compare.png)


## Do RES / NRES form two clusters in $(t_1, t_2)$?

**Partly — and the hard cities tell you *how* the blobs fail.**

Orange = non-residential, blue = residential. Globally the mass separates along **$t_1$** (NRES left, RES right); the $P=0.5$ cut is nearly vertical, so $t_2$ is a thin residual. Marginals: RES peaks near $t_1 \approx +4$, NRES near $t_1 \approx -2.5$, with a real overlap band around $t_1 \in [-1, +2]$.

| Probe on pooled OOF scores | macro-F1 |
|---|---:|
| 1D LR on $t_1$ only | 0.778 |
| 2D LR on $(t_1, t_2)$ | 0.783 |

Pooled-OOF F1 is descriptive. Honest LORO at k=2 is ~0.74 — transfer is harder than a global 2D cut.

```bash
.venv/bin/python -u scripts/plot_week4_pls_t1_t2.py
```

![Global t1 vs t2](../../outputs/week4_pls_loro/figures/pls_t1_t2_global.png)

![Marginals](../../outputs/week4_pls_loro/figures/pls_t1_t2_marginals.png)

### Reading the per-metro panels with precision / recall

What “two clear clusters” looks like in $(t_1,t_2)$ is the same signal the OOF LR metrics quantify. Join in-panel 2D structure with baseline-LR Prec/Recall/FN (RES = positive):

| Metro | in-panel 2D F1 | mean $t_1$ RES | mean $t_1$ NRES | Prec RES | Recall RES | FN rate | What the plot is doing |
|---|---:|---:|---:|---:|---:|---:|---|
| SF_W | 0.688 | -1.26 | -2.77 | 0.763 | 0.490 | **0.510** | FN-heavy (RES→NRES) |
| NYC_NE | 0.644 | -0.55 | -1.34 | 0.661 | 0.715 | **0.285** | FP-heavy (NRES→RES) |
| Chicago_MW | 0.725 | -0.12 | -1.83 | 0.814 | 0.562 | **0.438** | FN-heavy (RES→NRES) |
| Boston_NE | 0.756 | 2.29 | -0.42 | 0.679 | 0.847 | **0.153** | FP-heavy (NRES→RES) |
| Seattle_NW | 0.864 | 4.56 | -0.46 | 0.706 | 0.940 | **0.060** | FP-heavy (NRES→RES) |
| Houston_S | 0.784 | 1.38 | -2.45 | 0.930 | 0.599 | **0.401** | FN-heavy (RES→NRES) |
| LA_W | 0.865 | 2.07 | -2.52 | 0.812 | 0.931 | **0.069** | clean split |
| Denver_W | 0.866 | 2.93 | -2.48 | 0.849 | 0.886 | **0.114** | clean split |
| Atlanta_SE | 0.892 | 4.23 | -0.89 | 0.840 | 0.909 | **0.091** | clean split |
| Phoenix_SW | 0.908 | 1.16 | -3.56 | 0.945 | 0.861 | **0.139** | clean split |

**How to read the blobs**

1. **Clean two-blob cities** (Phoenix, Atlanta, Denver, LA) — class means on $t_1$ sit far apart (often $\Delta t_1 \gtrsim 4$). FN rate is low and Prec RES stays high. The black cut sits in a low-density gap; little ink in the wrong lobe.
2. **FN-heavy cities** (SF FN=0.51, Chicago 0.44, Houston 0.40) — a large blue mass still sits on the **left / NRES side** of $t_1$, or the RES cloud is pulled left so many true RES cells fall past the cut. Plot cue: blue points mixed into the orange lobe; high FN, often still decent Prec RES (model is shy about calling RES). Chicago’s RES mean $t_1$ is near **0** — almost no rightward RES lobe.
3. **FP-heavy cities** (NYC, Boston, and Seattle on metrics) — orange bleeds **right** into the RES side. Prec RES drops and/or NRES recall is weak. Plot cue: orange points across the cut into the blue region; FN rate alone can look “ok” (Boston 0.15, Seattle 0.06) while precision is the problem.
4. **NYC / SF / Chicago** are the visually overlapped panels — not one failure mode. SF/Chicago miss RES; NYC overlaps both ways with low Prec.

![Per metro](../../outputs/week4_pls_loro/figures/pls_t1_t2_by_region.png)

**Bottom line for the cluster plots:** two modes exist along $t_1$, but hard metros are not “no clusters” — they are **shifted / overlapping modes** whose failure type (FN vs FP) you can see as blue-on-the-left vs orange-on-the-right.


## Held-out 3D clusters after the within-region loss

Each panel is that metro's raw encoder output on the tiles the fold did not train on. The axes are encoder 1, 2, and 3. Blue is residential. Orange is non-residential. The gap is the distance between the two class centers divided by the mean radius. A gap of 3 is a clear separation. A gap below 1 is one cloud. v3 is the previous contrastive run.

| Metro | v4 F1 | v3 F1 | v4 gap | v3 gap |
|---|---:|---:|---:|---:|
| Atlanta | 0.898 | 0.840 | 3.51 | 2.34 |
| Phoenix | 0.878 | 0.849 | 3.19 | 2.43 |
| Denver | 0.862 | 0.852 | 3.15 | 2.81 |
| Seattle | 0.824 | 0.787 | 2.81 | 2.29 |
| Los Angeles | 0.793 | 0.842 | 2.50 | 2.39 |
| Houston | 0.709 | 0.711 | 1.96 | 1.65 |
| Chicago | 0.553 | 0.622 | 1.32 | 1.09 |
| Boston | 0.693 | 0.698 | 1.30 | 1.01 |
| San Francisco | 0.584 | 0.595 | 0.99 | 0.76 |
| New York | 0.586 | 0.552 | 0.63 | 0.35 |

![v4 versus v3 gap](../../outputs/week4_vit_100m_3d_v4/figures/v4_vs_v3_gap.png)

### Atlanta

![3D Atlanta](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Atlanta_SE.png)

### Phoenix

![3D Phoenix](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Phoenix_SW.png)

### Denver

![3D Denver](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Denver_W.png)

### Seattle

![3D Seattle](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Seattle_NW.png)

### Los Angeles

![3D Los Angeles](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_LA_W.png)

### Houston

![3D Houston](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Houston_S.png)

### Chicago

![3D Chicago](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Chicago_MW.png)

### Boston

![3D Boston](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Boston_NE.png)

### San Francisco

![3D San Francisco](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_SF_W.png)

### New York

![3D New York](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_NYC_NE.png)

Each fold has its own axes. The coordinates are not pooled across metros.


## Validation clusters while New York is held out

These are the tiles the NYC fold used to decide when to stop. They are a 10% split of the other nine metros, embedded with the checkpoint that had the highest validation gap (epoch 25, mean gap 3.33). The same fold's held-out New York gap is 0.63.

The four widest validation gaps are shown, then San Francisco, the narrowest in this split. Blue is residential. Orange is non-residential.

| Metro | Validation gap | Tiles |
|---|---:|---:|
| Phoenix | 5.44 | 514 |
| Atlanta | 4.98 | 367 |
| Denver | 3.74 | 743 |
| Los Angeles | 3.71 | 1,068 |
| Seattle | 3.59 | 225 |
| Houston | 2.80 | 637 |
| Chicago | 2.01 | 505 |
| Boston | 1.90 | 550 |
| San Francisco | 1.82 | 111 |

### Phoenix, validation gap 5.44

![Phoenix validation](../../outputs/week4_vit_100m_3d_v4/figures/val_nycfold_Phoenix_SW.png)

### Atlanta, validation gap 4.98

![Atlanta validation](../../outputs/week4_vit_100m_3d_v4/figures/val_nycfold_Atlanta_SE.png)

### Denver, validation gap 3.74

![Denver validation](../../outputs/week4_vit_100m_3d_v4/figures/val_nycfold_Denver_W.png)

### Los Angeles, validation gap 3.71

![Los Angeles validation](../../outputs/week4_vit_100m_3d_v4/figures/val_nycfold_LA_W.png)

### San Francisco, validation gap 1.82

![San Francisco validation](../../outputs/week4_vit_100m_3d_v4/figures/val_nycfold_SF_W.png)

The classes sit at opposite ends of a thin line. Phoenix and Atlanta have a visible gap along that line. San Francisco's two ends still touch. This is the training-metro split, not the held-out city.


## Per-metro precision / recall / FN rate

Positive class = **residential**. FN rate = missed RES / all true RES $= 1 - \mathrm{recall}_{\mathrm{res}}$. Same numbers used to annotate the cluster panels above.

Baseline LR OOF (LORO), sorted by RES FN rate:

| Metro | Prec RES | Recall RES | FN rate RES | Prec NRES | Recall NRES | macro-F1 | |
|---|---:|---:|---:|---:|---:|---:|---|
| SF_W | 0.763 | 0.490 | **0.510** | 0.624 | 0.848 | 0.658 | ← hard
| Chicago_MW | 0.814 | 0.562 | **0.438** | 0.666 | 0.872 | 0.710 | ← hard
| Houston_S | 0.930 | 0.599 | **0.401** | 0.704 | 0.955 | 0.770 |
| NYC_NE | 0.661 | 0.715 | **0.285** | 0.690 | 0.633 | 0.673 | ← hard
| Boston_NE | 0.679 | 0.847 | **0.153** | 0.797 | 0.599 | 0.719 | ← hard
| Phoenix_SW | 0.945 | 0.861 | **0.139** | 0.873 | 0.950 | 0.905 |
| Denver_W | 0.849 | 0.886 | **0.114** | 0.881 | 0.843 | 0.864 |
| Atlanta_SE | 0.840 | 0.909 | **0.091** | 0.901 | 0.827 | 0.868 |
| LA_W | 0.812 | 0.931 | **0.069** | 0.920 | 0.785 | 0.857 |
| Seattle_NW | 0.706 | 0.940 | **0.060** | 0.910 | 0.608 | 0.768 |

| Group | mean FN rate RES | mean Recall RES | mean Prec RES | mean macro-F1 |
|---|---:|---:|---:|---:|
| Hard (NYC, SF, Chicago, Boston) | 0.347 | 0.653 | 0.729 | 0.690 |
| Other | 0.145 | 0.855 | 0.847 | 0.839 |

![Per-metro Prec / Recall / FN](../../outputs/week4_pls_loro/figures/per_region_prec_recall_fnr.png)


In [ ]:
pr = pd.read_csv(OUT / "per_region_precision_recall.csv")
pr.loc[pr.model == "baseline_lr", [
    "region", "precision_res", "recall_res", "fn_rate_res",
    "precision_nres", "recall_nres", "macro_f1", "fn", "fp"
]].sort_values("fn_rate_res", ascending=False)


## Equation: first 10 PLS-DA components → P(residential)

Fold-averaged PLS-DA → logistic (LORO). Score means $\approx 0$, so $z_k \approx t_k / s_k$.

$$
\tilde{x}_j = \frac{A_j-\mu_j}{\sigma_j},\qquad t_k=\mathbf{r}_k^\top\tilde{\mathbf{x}},\quad k=1,\ldots,10
$$

$$
\mathrm{logit}=+0.0952+0.7071 t_{1}+0.3750 t_{2}+0.2715 t_{3}+0.2357 t_{4}+0.3275 t_{5}+0.2488 t_{6}+0.1374 t_{7}+0.0863 t_{8}+0.1481 t_{9}+0.1426 t_{10}
$$

$$
P(\mathrm{RES}\mid\mathbf{x})=\sigma(\mathrm{logit})=\frac{1}{1+e^{-\mathrm{logit}}}
$$

Numerically (same thing):

```
logit = +0.0952 +0.7071*t1 +0.3750*t2 +0.2715*t3 +0.2357*t4 +0.3275*t5 +0.2488*t6 +0.1374*t7 +0.0863*t8 +0.1481*t9 +0.1426*t10
P_res = 1 / (1 + exp(-logit))
```

OOF macro-F1 (fold-wise $k=10$): **0.793**.

Full $\mathbf{r}_k$ (64×10), $\boldsymbol{\mu}$, $\boldsymbol{\sigma}$, and compact $\mathbf{c}$ on scaled AE: `outputs/week4_pls_loro/pls_da_k10_*.{json,csv,md}`.

| component | $\beta_k$ (on $z_k$) | $s_k$ | coef on $t_k$ ($\beta_k/s_k$) |
|---:|---:|---:|---:|
| t1 | 2.2401 | 3.1681 | 0.7071 |
| t2 | 0.7896 | 2.1057 | 0.3750 |
| t3 | 0.5723 | 2.1076 | 0.2715 |
| t4 | 0.3740 | 1.5867 | 0.2357 |
| t5 | 0.4173 | 1.2740 | 0.3275 |
| t6 | 0.3399 | 1.3660 | 0.2488 |
| t7 | 0.2117 | 1.5413 | 0.1374 |
| t8 | 0.1349 | 1.5641 | 0.0863 |
| t9 | 0.1560 | 1.0533 | 0.1481 |
| t10 | 0.1453 | 1.0188 | 0.1426 |


In [ ]:
betas = pd.read_csv(OUT / "pls_da_k10_lr_betas.csv")
R = pd.read_csv(OUT / "pls_da_k10_loadings_R.csv", index_col=0)
c = pd.read_csv(OUT / "pls_da_k10_compact_c.csv")
print(betas.to_string(index=False))
print("\nTop |c| on scaled AE:")
print(c.reindex(c.c.abs().sort_values(ascending=False).index).head(12).to_string(index=False))


## Load outputs


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("../..").resolve()
OUT = ROOT / "outputs" / "week4_pls_loro"
summary = json.loads((OUT / "summary.json").read_text())
sweep = pd.read_csv(OUT / "sweep_summary.csv")
metrics = pd.read_csv(OUT / "loro_metrics.csv")
fine = json.loads((OUT / "fine_k_sweep.json").read_text())
fine_df = pd.read_csv(OUT / "fine_k_sweep.csv")
print("baseline LORO F1:", round(fine["baseline_loro_f1"], 4))
print("min-k criteria:", json.dumps(fine["min_k_criteria"], indent=2))
print("cov cumulants: 90%@k=%s  95%@k=%s  99%@k=%s" % (
    fine["cov_cum_k90"], fine["cov_cum_k95"], fine["cov_cum_k99"]))


## Minimum orthogonal dimensions (fine PLS-DA sweep)

PLS-DA scores are **orthogonal**. Fine LORO sweep over `k = 1..20` on the same balanced pure cells (8 GPUs).

```bash
.venv/bin/python -u scripts/fine_week4_pls_k_sweep.py
```

### Covariance with \(y\) (signal concentration)

Almost all supervised covariance sits in the first few scores:

| Cumulative cov²(t, y) | k |
|---|---:|
| ~90% | **1** |
| ~95% | **2** |
| ~99% | **5** |

### Classifier parity vs 64-D LR (LORO F1 = 0.7792)

| Criterion | Min k | F1 | gap |
|---|---:|---:|---:|
| within 0.02 F1 | **9** | 0.7612 | 0.0180 |
| within 0.01 F1 | **16** | 0.7708 | 0.0084 |
| within 2% relative | **11** | 0.7673 | 0.0119 |
| within 1% relative | **17** | 0.7723 | 0.0069 |

### Selected k on the fine curve

| k | LORO F1 | gap to 64-D | rel gap |
|---:|---:|---:|---:|
| 1 | 0.735 | 0.0444 | 0.0569 |
| 2 | 0.738 | 0.0415 | 0.0532 |
| 5 | 0.749 | 0.0302 | 0.0388 |
| 9 | 0.761 | 0.0180 | 0.0231 |
| 11 | 0.767 | 0.0119 | 0.0153 |
| 12 | 0.765 | 0.0137 | 0.0176 |
| 16 | 0.771 | 0.0084 | 0.0108 |
| 17 | 0.772 | 0.0069 | 0.0088 |
| 20 | 0.776 | 0.0029 | 0.0037 |
| 64 | 0.779 | 0.0000 | 0.0000 |

**Recommendation**
- Absolute minimum for the residential *signal*: **k = 5** (99% of cov with \(y\)).
- Best compression that still races LR: **k = 9–12**.
- Near-parity with full 64-D LR: **k = 16**.

![Fine k minimum](../../outputs/week4_pls_loro/figures/fine_k_minimum.png)


In [ ]:
fine_df


### Binary heads only (coarse sweep)


In [ ]:
bin_models = ["baseline_lr", "pls_da_lr", "pls_frac_lr", "pca_lr"]
sweep.loc[sweep.model.isin(bin_models), [
    "model", "n_components", "loro_macro_f1_mean", "loro_macro_f1_std"
]].sort_values(["model", "n_components"])


### Fraction heads (coarse sweep)


In [ ]:
frac_models = ["baseline_ridge", "pls_frac"]
sweep.loc[sweep.model.isin(frac_models), [
    "model", "n_components", "loro_mae_mean", "loro_mae_std"
]].sort_values(["model", "n_components"])


## Takeaways

1. **AE bands are collinear** (not orthogonal; effective rank ~10). PLS scores *are* orthogonal — use those as the compressed features.
2. **PLS ≈ full LR / Ridge** at enough components (Δ OOF F1 ≈ −0.001; fraction MAE slightly better at k=12). PCA at the same width lags → the gain is supervised, not just dim reduction.
3. **Minimum orthogonal set**
   - signal (99% cov with \(y\)): **k = 5**
   - race LR (within 0.02 F1): **k = 9–12**
   - near-parity (within 0.01 F1): **k = 16**
4. Artifacts under `outputs/week4_pls_loro/`: OOF preds, best-k PLS scores (`oof_pls_da_scores_k32.npy`), fold checkpoints (loadings/rotations), coarse + fine sweeps, figures.


In [ ]:
print("paths:", json.dumps(summary.get("paths", {}), indent=2))
print("backend:", summary.get("backend"), "gpus:", summary.get("n_gpus"), "cpus:", summary.get("n_cpus"))
print("elapsed_s:", summary.get("elapsed_s"))
if (OUT / "fine_k_sweep.json").exists():
    print("fine_k_sweep:", OUT / "fine_k_sweep.json")
    print("fine_k_figure:", OUT / "figures" / "fine_k_minimum.png")
